# Notebook 00 — Environment Check

**FIN 4600 · Lab 3 · Financial Data Analytics**

Run this notebook first. It does no finance — its only job is to prove that
your Python environment, your packages, and the data files are all working
before you invest time in the real labs.

**How to run a notebook cell:** click inside the cell and press
`Shift + Enter`. The `[ ]` to the left of the cell turns into `[*]` while it
runs and then into a number when it finishes.

If the very first cell fails, stop and go back to `SETUP.md`. Nothing further
in this course will work until it passes.

**How this lab works now.** Every code cell in this lab is complete and
ready to run. Your job is to run each cell in order, read the comments in the
code and the explanation above each cell, and check that what you see matches
what the text says you should see. A short Canvas comprehension quiz for each
notebook follows.

## 1. Which Python am I actually running?

This is the single most common source of confusion. Your computer may have
several Pythons installed. The one this notebook uses must be the one inside
the project's `.venv` folder, which is the environment `uv sync` built.

**What the next cell does.** It prints the full path of the Python program
that is running this notebook, and its version number, then checks whether
that path contains `.venv`.

#### Background: Python modules, `sys` and `pathlib`

- A **module** is a file of ready-made Python code. `import sys` loads it and
  gives you a name (`sys`) to reach everything inside it. `from pathlib import
  Path` loads just one thing, `Path`, from the `pathlib` module. Both modules
  ship with Python itself (the *standard library*), so nothing extra needs
  installing.
- **`sys`** describes the running interpreter. `sys.executable` is the path of
  the `python.exe` behind this notebook; `sys.version` is a text string that
  starts with the version number, e.g. `3.12.14 (main, ...)`.
- **`pathlib.Path`** treats a file path as an object rather than a string.
  You join pieces with `/` (`REPO / "data" / "file.csv"`), and it works the
  same on Windows (`\`) and macOS (`/`). It also has handy methods: `.exists()`,
  `.resolve()` (turn a relative path into a full one), `.parents`, `.glob()`.

#### Background: the `.venv` and the notebook *kernel*

A notebook does not run code itself. It sends each cell to a separate
Python process called the **kernel**, and shows you what comes back. Which
Python that kernel is decides which packages are available.

The project's **virtual environment** (`.venv`, built by `uv sync`) is a
private copy of Python with exactly the package versions this lab was written
against. Keeping each project in its own environment means upgrading pandas
for one course cannot break another. If the kernel is not the `.venv` one,
`import pandas` may fail, or worse, succeed with a different version that
gives slightly different numbers.

**What to look for:** a path ending in `.venv\Scripts\python.exe` (Windows)
or `.venv/bin/python` (macOS), and the `OK` line.

In [ ]:
# sys gives access to facts about the running Python interpreter (its path, its version)
import sys
# Path (from the standard-library pathlib module) represents a file or folder location as an object
from pathlib import Path

# Print a heading so the output is easy to read
print("Python executable:")
# sys.executable is the full path of the python.exe that is running this notebook's kernel
print("   ", sys.executable)
# sys.version is a long string such as "3.12.14 (main, ...)"; .split() breaks it at spaces and [0] keeps only the number
print("Python version:", sys.version.split()[0])

# If the interpreter's path contains ".venv", the kernel is the project's own virtual environment
if ".venv" in sys.executable:
    # Confirm the good case; "\n" at the start prints a blank line first
    print("\nOK — you are running the project virtual environment.")
# Otherwise the notebook is attached to some other Python that may lack the right packages
else:
    # One print() call; the adjacent string pieces below are glued together into a single message
    print(
        "\nWARNING — this does NOT look like the project's .venv.\n"  # first line of the warning
        "In VS Code, click 'Select Kernel' in the top-right of this notebook\n"  # how to fix it
        "and choose the interpreter whose path contains '.venv'.\n"  # which interpreter to pick
        "See the troubleshooting section of SETUP.md."  # where to read more
    )

## 2. Are the packages installed, and at what versions?

Recording versions matters more in finance than in most fields. If a risk
number cannot be reproduced, it cannot be audited — and a number that cannot
be audited is not a number a regulator will accept.

**What the next cell does.** It imports the five third-party packages the
lab depends on and prints each one's version number.

#### Background: the five packages this lab uses

- **NumPy** — fast arrays of numbers and the maths on them (means, matrix
  algebra, random numbers). Almost every other scientific Python library is
  built on top of it.
- **pandas** — labelled tables (`DataFrame`) with dates, joins, group-by and
  time-series tools. It was created at a hedge fund (AQR) in 2008 for exactly
  the kind of market-data work in this lab, and is now standard across banks,
  asset managers and fintechs.
- **matplotlib** — the base plotting library; you control every axis, label
  and line.
- **seaborn** — statistical charts (heatmaps, distributions) built on top of
  matplotlib with less code.
- **scikit-learn** (imported as `sklearn`) — the standard machine-learning
  toolkit: models, train/test splits, cross-validation and metrics, all with
  the same `fit` / `predict` interface.

Every Python module carries its version in an attribute called `__version__`
(two underscores each side), and its own name in `__name__`.

**What to look for:** five lines, one per package, each with a version
number. An `ImportError` instead means the kernel is not the `.venv`.

In [ ]:
# Load each package; an ImportError on any of these lines means the kernel is not the project's .venv
import numpy
# pandas: labelled tables (DataFrames) and time-series tools
import pandas
# matplotlib: the base plotting library
import matplotlib
# seaborn: statistical charts built on matplotlib
import seaborn
# scikit-learn's import name is "sklearn"
import sklearn

# Loop over the five module objects held in a tuple, one at a time
for module in (numpy, pandas, matplotlib, seaborn, sklearn):
    # f-string: {module.__name__:<14} left-aligns the name in a 14-character column, then the version string follows
    print(f"{module.__name__:<14} {module.__version__}")

## 3. Can I find the data?

Every notebook in this lab locates the repository by walking up the folder
tree until it finds `data/sp500_prices.csv`. That makes the notebooks work no
matter which folder VS Code happens to start them in.

**What the next cell does.** It defines a small function, `find_repo_root`,
that starts in the current folder and climbs upward one parent folder at a
time until it finds a folder containing `data/sp500_prices.csv`. It then
lists every CSV file in `data/` with its size.

**How the search works.** `Path.cwd()` is the folder the kernel started in.
`.resolve()` turns it into a full, absolute path. `.parents` is the list of
folders above it (parent, grandparent, ... up to the drive root). The loop
tests each candidate with `.exists()` and returns the first match. If none
matches it *raises* an error with a helpful message, rather than carrying on
with a wrong path. Every later notebook pastes in this same function.

`DATA.glob("*.csv")` returns every file in `data/` whose name ends `.csv`;
`.stat().st_size` is a file's size in bytes, so dividing by 1024 gives
kilobytes.

**What to look for:** the repository folder path and five data files:
`german_credit.csv`, `sp500_companies.csv`, `sp500_prices.csv`,
`us10y_monthly.csv` and `vix_daily.csv`.

In [ ]:
# Define a reusable function; start=None means "if no folder is given, use the current one"
def find_repo_root(start=None):
    """Return the repository root — the folder that contains data/sp500_prices.csv."""  # docstring: what the function returns
    # Use the given folder or the kernel's current working folder, made into a full absolute path
    here = Path(start or Path.cwd()).resolve()
    # Try this folder first, then each folder above it (*here.parents unpacks parent, grandparent, ...)
    for candidate in [here, *here.parents]:
        # The / operator joins path pieces; .exists() is True if that file is really on disk
        if (candidate / "data" / "sp500_prices.csv").exists():
            # Found it: hand this folder back to the caller and stop searching
            return candidate
    # Reached the top of the drive without a match: stop with a clear, actionable error message
    raise FileNotFoundError(
        "Could not find the repository root. In VS Code use File > Open Folder "  # message, part 1
        "and open the mtu4600-lab3-analytics folder itself, then re-run."  # part 2, glued onto part 1
    )


# Run the search once and keep the answer; REPO is a Path object for the project folder
REPO = find_repo_root()
# Path of the data folder inside the repository
DATA = REPO / "data"

# Show where the repository was found
print("Repository root:", REPO)
# Heading for the file listing ("\n" adds a blank line first)
print("\nData files found:")
# glob("*.csv") finds every CSV in data/; sorted() puts them in alphabetical order
for csv in sorted(DATA.glob("*.csv")):
    # st_size is the file size in bytes; dividing by 1024 converts to kilobytes
    size_kb = csv.stat().st_size / 1024
    # :<24 left-aligns the file name in 24 characters; :8.1f prints the size 8 wide with 1 decimal place
    print(f"   {csv.name:<24} {size_kb:8.1f} KB")

## 4. Can I load the data and draw a chart?

**What the next cell does.** It reads the daily price file into a pandas
table, prints its size, date range and number of tickers, and shows the first
five rows. Notebook 01 explains every one of these pandas tools in depth;
here we only need to see that they work.

- `pd.read_csv(path, parse_dates=["date"])` reads a comma-separated text file
  into a **DataFrame** (a table with named columns) and converts the `date`
  column from text into real dates.
- `plt.style.use(...)` loads the course's shared chart style (`fin4600.mplstyle`)
  so every chart in the lab has the same fonts and colours.
- A notebook cell displays the value of its **last line** automatically, which
  is why `prices.head()` shows a formatted table without a `print`.

**What to look for:** about 37,770 rows, dates from 2013-02-08 to 2018-02-07,
30 tickers, and a table starting with Apple (AAPL).

In [ ]:
# Import pandas under its conventional short name "pd"
import pandas as pd
# pyplot is matplotlib's chart-drawing interface; "plt" is the conventional short name
import matplotlib.pyplot as plt

# Apply the course chart style file (fonts, colours, gridlines) to every chart drawn from now on
plt.style.use(REPO / "fin4600.mplstyle")

# Read the CSV into a DataFrame; parse_dates turns the "date" text column into real datetime values
prices = pd.read_csv(DATA / "sp500_prices.csv", parse_dates=["date"])
# len() of a DataFrame is its row count; the :, format adds thousands separators (37,770)
print("Loaded", f"{len(prices):,}", "rows of daily prices")
# .min()/.max() of a date column give the earliest/latest date; .date() drops the time-of-day part
print("Date range:", prices["date"].min().date(), "to", prices["date"].max().date())
# nunique() counts how many different ticker symbols appear
print("Tickers:", prices["ticker"].nunique())

# head() returns the first 5 rows; as the last line of the cell, Jupyter displays it as a table
prices.head()

**What the next cell does.** It keeps only Apple's rows, makes the date the
row label (the *index*), and draws a line chart of the closing price.

#### Background: matplotlib figures and axes

matplotlib separates the **figure** (the whole image) from the **axes** (one
plotting area inside it, with its own x and y axis). `fig, ax = plt.subplots()`
creates one of each. You then call methods on `ax`: `ax.plot(x, y)` draws a
line, `ax.set_title`, `ax.set_ylabel` and `ax.set_xlabel` add labels, and
`plt.show()` renders the finished picture. This "create the axes, then
decorate it" pattern is used for every chart in the lab.

`prices["ticker"] == "AAPL"` produces a column of True/False values, one per
row; `prices.loc[...]` keeps the rows where it is True. Notebook 01 calls this
a **boolean mask**.

**What to look for:** a chart of Apple's price rising from about \$68 in February
2013 to about \$160 by February 2018.

In [ ]:
# Keep only Apple's rows (boolean mask inside .loc), then make the date column the row index for plotting
aapl = prices.loc[prices["ticker"] == "AAPL"].set_index("date")

# Create a figure (the whole picture) and one axes (the plotting area) inside it
fig, ax = plt.subplots()
# Draw a line: x = the dates in the index, y = the closing price column
ax.plot(aapl.index, aapl["close"])
# Chart title
ax.set_title("Apple (AAPL) daily closing price")
# Label the y axis with its units
ax.set_ylabel("Price (USD)")
# Leave the x axis unlabelled; the dates speak for themselves
ax.set_xlabel("")
# Render the chart below the cell
plt.show()

## 5. Is scikit-learn working?

A throwaway model on made-up numbers — we only want to see that the library
imports, fits, and predicts.

**What the next cell does.** It makes 200 rows of random numbers with three
columns, defines a yes/no label from them by a simple rule, fits a logistic
regression, and prints the share of rows it classifies correctly.

- `np.random.default_rng(42)` creates a random-number generator. The **seed**
  `42` makes the "random" numbers identical every time the cell runs, so
  everyone in the class gets the same result. Reproducibility again.
- `rng.normal(size=(200, 3))` draws a 200-row, 3-column array of standard
  normal numbers (mean 0, standard deviation 1).
- `X[:, 0]` means "all rows, column 0". The label `y` is 1 when
  `column0 + 0.5 * column1` is positive, else 0.
- **Logistic regression** is a classification model that turns a weighted sum
  of the inputs into a probability between 0 and 1. Notebook 04 covers it
  properly. Here it is a smoke test: `.fit(X, y)` learns the weights, and
  `.score(X, y)` returns the accuracy (fraction correct) on the same rows.

**What to look for:** accuracy close to 1 (the baseline run printed 0.99).
That is expected and *not* impressive: the label was built from a straight-line
rule, which is exactly what logistic regression can learn, and we are scoring
on the training data itself.

In [ ]:
# NumPy, under its conventional short name "np", for arrays and random numbers
import numpy as np
# LogisticRegression is scikit-learn's yes/no (binary) classification model
from sklearn.linear_model import LogisticRegression

# A random-number generator with a fixed seed, so every run draws exactly the same numbers
rng = np.random.default_rng(42)
# 200 rows x 3 columns of standard-normal random numbers: these are the made-up "features"
X = rng.normal(size=(200, 3))
# Label = 1 if column 0 + half of column 1 is positive, else 0; .astype(int) turns True/False into 1/0
y = (X[:, 0] + 0.5 * X[:, 1] > 0).astype(int)

# Create the model and fit (learn) its weights from X and y in one line
model = LogisticRegression().fit(X, y)
# score() returns accuracy (share of rows classified correctly); round to 3 decimal places
print("Training accuracy on synthetic data:", round(model.score(X, y), 3))

---

## If every cell above ran without an error, you are ready.

Move on to `01_pandas_fundamentals.ipynb`.

### Submitting this notebook

1. Save the notebook (`Ctrl/Cmd + S`).
2. In the VS Code **Source Control** panel, stage the change, write a commit
   message such as `Completed environment check`, and click **Commit**.
3. Click **Sync Changes** to push it to your own GitHub repository.


### Before you move on

Take the short **Canvas comprehension quiz for Notebook 00**. It checks that
you understood what each cell above did and why.